# Hackathon Digital Twin

This notebook talks to the hackathon microscope like a real instrument. Move the beam, acquire a camera image, then read the returned Tiled key.

Start the servers from the repo root before running the notebook:

```bash
uv run python startup_scripts/run_servers.py --yaml configs/hackathon.yaml --microscope dt
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import tango
from tiled.client import from_uri

microscope = tango.DeviceProxy('asyncroscopy/microscope/default')
microscope.set_timeout_millis(120_000)

client = from_uri('http://127.0.0.1:9091')

In [ ]:
source_path = Path('/Users/austin/Downloads/18167694/hackathon_data/hackathon_camera_source.h5')
source_path.exists(), source_path

Beam positions are fractional coordinates in `[0, 1]`. The digital twin maps them onto the first two dimensions of the 4D-STEM source dataset.

In [ ]:
microscope.place_beam([0.50, 0.50])
key = microscope.acquire_camera_image()
key

In [ ]:
image = client[key]['image'].read()
metadata = dict(client[key]['image'].metadata)

image.shape, image.dtype, metadata

In [ ]:
plt.figure(figsize=(5, 5))
plt.imshow(image, cmap='gray')
plt.axis('off');

Try a few beam positions and compare the returned camera frames.

In [ ]:
positions = [(0.10, 0.10), (0.30, 0.70), (0.80, 0.40)]
fig, axes = plt.subplots(1, len(positions), figsize=(12, 4))

for ax, position in zip(axes, positions):
    microscope.place_beam(position)
    key = microscope.acquire_camera_image()
    image = client[key]['image'].read()
    beam_index = dict(client[key]['image'].metadata).get('beam_index')
    ax.imshow(image, cmap='gray')
    ax.set_title(f'{position}\n{beam_index}')
    ax.axis('off')

plt.tight_layout()